# SVM, Decision Tree, Random Forest e Gradient Boosting em 6 datasets

Notebook gerado a partir do repositório (`scripts/gerar_notebook.py`): o código abaixo é exatamente
o do pacote `src/classificadores`.

**Como usar no Colab:** *Ambiente de execução → Executar tudo*. Os CSVs são baixados da API pública
do Kaggle (sem login) e do GitHub. A execução completa leva cerca de 10 a 15 minutos.

O que o notebook faz, para cada um dos 6 datasets:

1. divide em treino (80%) e teste (20%), estratificado;
2. percorre os 4 classificadores, guardados num dicionário, com um único laço `for`;
3. otimiza os hiperparâmetros de cada um com `RandomizedSearchCV` e validação cruzada estratificada de 5 folds;
4. mede acurácia, precisão, recall, F1, ROC AUC e matriz de confusão no teste;
5. compara treino x validação e as curvas de aprendizado para diagnosticar **alto viés** ou **alta variância**,
   e justifica a escolha entre **Random Forest** e **Gradient Boosting**.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

## 1. Configuração: constantes e os 6 datasets

Configuração geral: constantes do experimento e descrição dos 6 datasets.

Cada dataset declara explicitamente de onde vem, qual coluna é o alvo e quais colunas
entram (``usar``) ou saem (``drop``). Nada é adivinhado: se o arquivo ou a coluna não
existir, o carregamento falha com uma mensagem clara.

In [ ]:
"""Configuração geral: constantes do experimento e descrição dos 6 datasets.

Cada dataset declara explicitamente de onde vem, qual coluna é o alvo e quais colunas
entram (``usar``) ou saem (``drop``). Nada é adivinhado: se o arquivo ou a coluna não
existir, o carregamento falha com uma mensagem clara.
"""


import numpy as np
import pandas as pd

RANDOM_STATE = 42
TAMANHO_TESTE = 0.20  # 80% treino / 20% teste, estratificado
N_FOLDS = 5  # folds da validação cruzada estratificada
N_ITER = 10  # combinações de hiperparâmetros sorteadas por modelo no RandomizedSearchCV
METRICA_BUSCA = "f1"  # F1 macro: critério de escolha dos hiperparâmetros (todas as classes pesam igual)

# GradientBoostingClassifier "clássico" leva ~1 min por ajuste no Fishmorph (uma árvore por classe a
# cada iteração). HistGradientBoostingClassifier é o mesmo algoritmo com histogramas, ~20x mais rápido.
USAR_HIST_GB = True

# Limiares do diagnóstico de viés e variância (seção "Metodologia" do README)
LIMIAR_GAP_VARIANCIA = 0.10  # F1 de treino - F1 de validação acima disso = alta variância
LIMIAR_TREINO_VIES = 0.80  # F1 de treino abaixo disso (e gap pequeno) = alto viés
LIMIAR_GAP_VIES = 0.05

KAGGLE_API = "https://www.kaggle.com/api/v1/datasets/download/{slug}/{arquivo}"
GITHUB_RAW = "https://raw.githubusercontent.com/{slug}/main/{arquivo}"


def prep_eleicoes(df: pd.DataFrame) -> pd.DataFrame:
    """Alvo: o republicano venceu de fato no estado? (1 = sim).

    1932 quase não tem resultado real no arquivo e cai fora aqui.
    """
    df = df.dropna(subset=["pct_real_rep", "pct_real_dem"]).copy()
    df["vencedor_real"] = (df["pct_real_rep"] > df["pct_real_dem"]).astype(int)
    return df


def prep_cardiopatia(df: pd.DataFrame) -> pd.DataFrame:
    """Colesterol e pressão iguais a 0 significam "não medido": viram NaN para o imputador."""
    df = df.copy()
    for col in ["Cholesterol", "RestingBP"]:
        df[col] = df[col].replace(0, np.nan)
    return df


DATASETS: dict[str, dict] = {
    "Câncer de Mama": {
        "slug_curto": "cancer_mama",
        "fonte": "kaggle",
        "slug": "ahmedesso/brest-cancer",
        "arquivo": "data.csv",
        "target": "diagnosis",  # M = maligno, B = benigno
        "classe_positiva": "M",
        "drop": ["id", "Unnamed: 32"],  # identificador + coluna vazia que vem no CSV
        "descricao": "Breast Cancer Wisconsin (Diagnostic): 30 medidas de núcleos celulares.",
    },
    "Eleições (Literary Digest)": {
        "slug_curto": "eleicoes",
        "fonte": "github",
        "slug": "Gabriel-SZ-Silva/LiteraryDigestElections",
        "arquivo": "dataset_literary_digest_consolidado_1924_1936.csv",
        "target": "vencedor_real",
        "classe_positiva": 1,
        "drop": [
            # Resultado real: o alvo é derivado dele (seria vazamento de dados)
            "votos_real_rep", "votos_real_dem", "total_real", "pct_real_rep", "pct_real_dem",
            "vies_republicano_pp",
            # Ano e candidatos: em cada eleição um partido venceu quase todos os estados,
            # então essas colunas entregariam a resposta sem olhar para a pesquisa
            "ano", "candidato_rep", "candidato_dem",
        ],
        "prep": prep_eleicoes,
        "descricao": "Pesquisa do Literary Digest (1924-1936) por estado; prevê o vencedor real.",
    },
    "Fishmorph (Ordem)": {
        # Vertente "Ordem". Para a vertente "Família" (197 classes), use
        # arquivo="FISHMORPH_Family_Dataset.csv" e target="Family".
        "slug_curto": "fishmorph",
        "fonte": "kaggle",
        "slug": "menegidio/fishmorph-dataset",
        "arquivo": "FISHMORPH_Order_Dataset.csv",
        "target": "Order",
        "drop": [],
        "min_por_classe": 10,  # ordens com < 10 espécies inviabilizam treino/teste/CV estratificados
        "descricao": "10 medidas morfológicas de 8342 espécies de peixes de água doce; prevê a ordem.",
    },
    "Cardiopatia": {
        "slug_curto": "cardiopatia",
        "fonte": "kaggle",
        "slug": "fedesoriano/heart-failure-prediction",
        "arquivo": "heart.csv",
        "target": "HeartDisease",
        "classe_positiva": 1,
        "drop": [],
        "prep": prep_cardiopatia,
        "descricao": "Heart Failure Prediction: 11 atributos clínicos de 918 pacientes.",
    },
    "Estresse Estudantil": {
        "slug_curto": "estresse",
        "fonte": "kaggle",
        "slug": "rxnach/student-stress-factors-a-comprehensive-analysis",
        "arquivo": "StressLevelDataset.csv",
        "target": "stress_level",  # 0 = baixo, 1 = médio, 2 = alto
        "drop": [],
        "descricao": "20 fatores psicológicos, físicos, ambientais e acadêmicos de 1100 estudantes.",
    },
    "Ultimate Pokémon": {
        "slug_curto": "pokemon",
        "fonte": "kaggle",
        "slug": "darkmatternet/ultimate-pokmon-dataset-2025",
        "arquivo": "pokemon_complete_2025.csv",
        "target": "type_1",  # tipo primário (18 classes)
        # Só atributos numéricos de batalha e biologia. Ficam de fora nome, descrição, sprite,
        # habilidades e type_2, que praticamente entregam o tipo.
        "usar": [
            "num_types", "hp", "attack", "defense", "sp_attack", "sp_defense", "speed",
            "base_stat_total", "height_m", "weight_kg", "base_experience", "capture_rate",
            "base_happiness", "hatch_counter", "gender_rate", "bmi", "attack_defense_ratio",
            "physical_total", "special_total", "offensive_total", "defensive_total",
        ],
        "descricao": "1025 Pokémon com atributos de batalha e biologia; prevê o tipo primário.",
    },
}

## 2. Download (com cache) e preparação dos dados

Download (com cache local) e preparação dos datasets.

In [ ]:
"""Download (com cache local) e preparação dos datasets."""


import io
import os
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd


PASTA_DADOS = Path(os.environ.get("CLASSIFICADORES_DADOS", "data/raw"))


def _baixar(url: str) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "classificadores-6-datasets"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        conteudo = resp.read()
    if conteudo[:2] == b"PK":  # o Kaggle às vezes entrega o CSV dentro de um .zip
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            conteudo = z.read(z.namelist()[0])
    return conteudo


def caminho_local(cfg: dict, pasta: Path | None = None) -> Path:
    return Path(pasta or PASTA_DADOS) / cfg["arquivo"]


def obter_csv(cfg: dict, pasta: Path | None = None) -> pd.DataFrame:
    """Lê o CSV do cache local; se não existir, baixa e guarda no cache.

    Ordem de tentativa: cache local -> download direto (API pública do Kaggle ou GitHub raw)
    -> kagglehub. Se tudo falhar, a mensagem diz onde colocar o arquivo manualmente.
    """
    local = caminho_local(cfg, pasta)
    if local.exists():
        return pd.read_csv(local)

    url = (GITHUB_RAW if cfg["fonte"] == "github" else KAGGLE_API).format(**cfg)
    erros = []
    try:
        conteudo = _baixar(url)
        local.parent.mkdir(parents=True, exist_ok=True)
        local.write_bytes(conteudo)
        return pd.read_csv(local)
    except Exception as e:  # noqa: BLE001
        erros.append(f"download direto ({url}): {e}")

    if cfg["fonte"] == "kaggle":
        try:
            import kagglehub

            pasta_kh = kagglehub.dataset_download(cfg["slug"])
            return pd.read_csv(Path(pasta_kh) / cfg["arquivo"])
        except Exception as e:  # noqa: BLE001
            erros.append(f"kagglehub: {e}")

    raise RuntimeError(
        f"Não consegui obter {cfg['arquivo']}. Baixe-o manualmente e salve em {local}. "
        f"Detalhes: {erros}"
    )


def preparar(df: pd.DataFrame, cfg: dict) -> tuple[pd.DataFrame, pd.Series]:
    """Aplica o pré-tratamento do dataset e separa X (atributos) e y (alvo)."""
    if cfg.get("prep"):
        df = cfg["prep"](df)

    alvo = cfg["target"]
    if alvo not in df.columns:
        raise KeyError(f"coluna-alvo '{alvo}' não encontrada. Colunas disponíveis: {list(df.columns)}")
    df = df.dropna(subset=[alvo])  # linhas sem rótulo não servem para treino nem teste

    minimo = cfg.get("min_por_classe")
    if minimo:
        contagem = df[alvo].value_counts()
        df = df[df[alvo].isin(contagem[contagem >= minimo].index)]

    if "usar" in cfg:
        faltando = [c for c in cfg["usar"] if c not in df.columns]
        if faltando:
            raise KeyError(f"colunas de entrada não encontradas: {faltando}")
        X = df[cfg["usar"]]
    else:
        X = df.drop(columns=[alvo] + [c for c in cfg.get("drop", []) if c in df.columns])
    return X.reset_index(drop=True), df[alvo].reset_index(drop=True)


def classes_removidas(df: pd.DataFrame, cfg: dict) -> pd.Series:
    """Classes descartadas por terem menos exemplos que ``min_por_classe``."""
    minimo = cfg.get("min_por_classe")
    if not minimo:
        return pd.Series(dtype=int)
    contagem = df[cfg["target"]].value_counts()
    return contagem[contagem < minimo]


def carregar_dataset(nome: str, pasta: Path | None = None) -> tuple[pd.DataFrame, pd.Series]:
    cfg = DATASETS[nome]
    return preparar(obter_csv(cfg, pasta), cfg)

## 3. Pré-processamento e os 4 classificadores, num único dicionário

Pré-processamento e os 4 classificadores, instanciados num único dicionário.

O resto do código percorre ``criar_modelos()`` com um laço ``for``: nenhum modelo tem
código de treino ou avaliação próprio.

In [ ]:
"""Pré-processamento e os 4 classificadores, instanciados num único dicionário.

O resto do código percorre ``criar_modelos()`` com um laço ``for``: nenhum modelo tem
código de treino ou avaliação próprio.
"""


import numpy as np
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import (
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier



def criar_preprocessador() -> ColumnTransformer:
    """Imputação + escala para numéricos; imputação + one-hot para categóricos.

    Fica dentro do Pipeline, então é ajustado só nos folds de treino (sem vazamento de dados).
    "Numérico / não numérico" cobre qualquer dtype (int8, bool, str do pandas 3...).
    """
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputacao", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), make_column_selector(dtype_include=np.number)),
            ("cat", Pipeline([
                ("imputacao", SimpleImputer(strategy="most_frequent")),
                # denso: o HistGradientBoosting não aceita matriz esparsa
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), make_column_selector(dtype_exclude=np.number)),
        ]
    )


def _pipeline(modelo) -> Pipeline:
    return Pipeline([("prep", criar_preprocessador()), ("modelo", modelo)])


IMPUTACAO = {"prep__num__imputacao__strategy": ["mean", "median"]}


def criar_modelos(usar_hist_gb: bool = USAR_HIST_GB) -> dict[str, dict]:
    """Dicionário nome -> {"pipeline": Pipeline, "espaco": espaço de busca de hiperparâmetros}."""
    if usar_hist_gb:
        gb = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
        espaco_gb = {
            **IMPUTACAO,
            "modelo__max_iter": randint(50, 300),
            "modelo__learning_rate": uniform(0.01, 0.2),
            "modelo__max_depth": [3, 5, 8, None],
            "modelo__l2_regularization": [0.0, 0.1, 1.0],
        }
    else:
        gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
        espaco_gb = {
            **IMPUTACAO,
            "modelo__n_estimators": randint(50, 150),
            "modelo__learning_rate": uniform(0.01, 0.2),
            "modelo__max_depth": randint(3, 8),
        }

    return {
        "Decision Tree": {
            "pipeline": _pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
            "espaco": {
                **IMPUTACAO,
                "modelo__max_depth": [3, 5, 10, 20, None],
                "modelo__min_samples_split": [2, 5, 10],
                "modelo__min_samples_leaf": [1, 2, 5],
                "modelo__class_weight": [None, "balanced"],
            },
        },
        "Random Forest": {
            "pipeline": _pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
            "espaco": {
                **IMPUTACAO,
                "modelo__n_estimators": randint(100, 300),
                "modelo__max_depth": [5, 10, 20, None],
                "modelo__max_features": ["sqrt", "log2"],
                "modelo__min_samples_leaf": [1, 2, 5],
                "modelo__class_weight": [None, "balanced"],
            },
        },
        "Gradient Boosting": {"pipeline": _pipeline(gb), "espaco": espaco_gb},
        "SVM": {
            "pipeline": _pipeline(SVC(cache_size=1000)),
            # gamma só existe no RBF; no kernel linear, C alto pode demorar muito para convergir
            # em dados não separáveis, por isso C <= 1 nele.
            "espaco": [
                {
                    **IMPUTACAO,
                    "modelo__kernel": ["rbf"],
                    "modelo__C": loguniform(0.1, 100),
                    "modelo__gamma": ["scale", 0.01, 0.1, 1],
                    "modelo__class_weight": [None, "balanced"],
                },
                {
                    **IMPUTACAO,
                    "modelo__kernel": ["linear"],
                    "modelo__C": loguniform(0.01, 1),
                    "modelo__class_weight": [None, "balanced"],
                },
            ],
        },
    }

## 4. Divisão treino/teste, validação cruzada automática e métricas

Divisão treino/teste, validação cruzada automática, busca de hiperparâmetros e métricas.

In [ ]:
"""Divisão treino/teste, validação cruzada automática, busca de hiperparâmetros e métricas."""


import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import label_binarize


METRICAS = ["accuracy", "precision", "recall", "f1", "roc_auc"]
NOMES_METRICAS = {
    "accuracy": "Acurácia",
    "precision": "Precisão",
    "recall": "Recall",
    "f1": "F1",
    "roc_auc": "ROC AUC",
}


# --------------------------------------------------------------------------------------
# Divisão e validação cruzada
# --------------------------------------------------------------------------------------
def dividir(X: pd.DataFrame, y: pd.Series):
    """Treino/teste 80/20, estratificado quando todas as classes têm ao menos 2 exemplos."""
    estratificar = y if y.value_counts().min() >= 2 else None
    return train_test_split(X, y, test_size=TAMANHO_TESTE, stratify=estratificar,
                            random_state=RANDOM_STATE)


def criar_cv(y_train: pd.Series):
    """StratifiedKFold com o maior número de folds (até N_FOLDS) que a menor classe permite."""
    menor = int(y_train.value_counts().min())
    if menor >= 2:
        return StratifiedKFold(n_splits=min(N_FOLDS, menor), shuffle=True, random_state=RANDOM_STATE)
    return KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)


# --------------------------------------------------------------------------------------
# Pontuações contínuas e ROC AUC (funciona com predict_proba ou decision_function)
# --------------------------------------------------------------------------------------
def pontuacoes(estimador, X) -> np.ndarray:
    """Pontuação contínua por classe, na ordem de ``estimador.classes_``.

    Usa ``predict_proba`` quando existe; no SVC (sem probability=True) usa ``decision_function``,
    o que basta para a ROC, que só depende da ordenação das pontuações.
    """
    if hasattr(estimador, "predict_proba"):
        try:
            return estimador.predict_proba(X)
        except AttributeError:
            pass
    s = estimador.decision_function(X)
    if s.ndim == 1:  # binário: decision_function devolve só a pontuação da classe positiva
        s = np.column_stack([-s, s])
    return s


def roc_auc(y_true, scores: np.ndarray, classes) -> float:
    """ROC AUC binária ou, no multiclasse, média macro um-contra-todos (OvR)."""
    classes = list(classes)
    if len(classes) == 2:
        return roc_auc_score(np.asarray(y_true) == classes[1], scores[:, 1])
    Y = label_binarize(y_true, classes=classes)
    presentes = Y.sum(axis=0) > 0  # classe ausente no fold não tem ROC definida
    return roc_auc_score(Y[:, presentes], scores[:, presentes], average="macro")


def _auc_scorer(estimador, X, y):
    return roc_auc(y, pontuacoes(estimador, X), estimador.classes_)


SCORING = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": _auc_scorer,
}


# --------------------------------------------------------------------------------------
# Resultado por modelo
# --------------------------------------------------------------------------------------
@dataclass
class ResultadoModelo:
    nome: str
    busca: RandomizedSearchCV
    tempo_s: float
    cv_treino: dict[str, float]  # média nos folds de treino da CV (melhores hiperparâmetros)
    cv_validacao: dict[str, float]  # média nos folds de validação da CV
    cv_validacao_std: dict[str, float]
    teste: dict[str, float]
    matriz_confusao: np.ndarray
    y_pred: np.ndarray
    scores_teste: np.ndarray

    @property
    def estimador(self):
        return self.busca.best_estimator_


@dataclass
class ResultadoDataset:
    nome: str
    classes: list
    n_linhas: int
    n_atributos: int
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    modelos: dict[str, ResultadoModelo] = field(default_factory=dict)
    curvas: dict = field(default_factory=dict)  # preenchido por diagnostico.curvas_aprendizado
    diagnostico: dict = field(default_factory=dict)


def metricas_teste(estimador, X_test, y_test) -> tuple[dict, np.ndarray, np.ndarray, np.ndarray]:
    y_pred = estimador.predict(X_test)
    scores = pontuacoes(estimador, X_test)
    m = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc(y_test, scores, estimador.classes_),
    }
    cm = confusion_matrix(y_test, y_pred, labels=estimador.classes_)
    return m, cm, y_pred, scores


def curva_roc(y_test, scores: np.ndarray, classes) -> tuple[np.ndarray, np.ndarray, float]:
    """FPR, TPR e AUC. No multiclasse, curva micro-média um-contra-todos."""
    classes = list(classes)
    if len(classes) == 2:
        fpr, tpr, _ = roc_curve(np.asarray(y_test) == classes[1], scores[:, 1])
    else:
        Y = label_binarize(y_test, classes=classes)
        fpr, tpr, _ = roc_curve(Y.ravel(), scores.ravel())
    return fpr, tpr, auc(fpr, tpr)


def avaliar_modelo(nome: str, item: dict, X_train, y_train, X_test, y_test, cv,
                   n_iter: int = N_ITER) -> ResultadoModelo:
    """Busca de hiperparâmetros com CV automática e avaliação no conjunto de teste."""
    inicio = time.time()
    busca = RandomizedSearchCV(
        estimator=item["pipeline"],
        param_distributions=item["espaco"],
        n_iter=n_iter,
        cv=cv,
        scoring=SCORING,
        refit=METRICA_BUSCA,
        return_train_score=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        error_score="raise",
    )
    busca.fit(X_train, y_train)
    i = busca.best_index_
    r = busca.cv_results_
    teste, cm, y_pred, scores = metricas_teste(busca.best_estimator_, X_test, y_test)
    return ResultadoModelo(
        nome=nome,
        busca=busca,
        tempo_s=time.time() - inicio,
        cv_treino={m: float(r[f"mean_train_{m}"][i]) for m in METRICAS},
        cv_validacao={m: float(r[f"mean_test_{m}"][i]) for m in METRICAS},
        cv_validacao_std={m: float(r[f"std_test_{m}"][i]) for m in METRICAS},
        teste=teste,
        matriz_confusao=cm,
        y_pred=y_pred,
        scores_teste=scores,
    )


def avaliar_dataset(nome: str, X: pd.DataFrame, y: pd.Series, modelos: dict, n_iter: int = N_ITER,
                    log=print) -> ResultadoDataset:
    X_train, X_test, y_train, y_test = dividir(X, y)
    cv = criar_cv(y_train)
    res = ResultadoDataset(nome=nome, classes=sorted(y.unique().tolist(), key=str),
                           n_linhas=len(X), n_atributos=X.shape[1],
                           X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)
    for nome_modelo, item in modelos.items():  # o mesmo código para os 4 classificadores
        rm = avaliar_modelo(nome_modelo, item, X_train, y_train, X_test, y_test, cv, n_iter)
        res.modelos[nome_modelo] = rm
        log(f"   ✓ {nome_modelo:<18} F1 CV = {rm.cv_validacao['f1']:.4f} | "
            f"F1 teste = {rm.teste['f1']:.4f} | AUC teste = {rm.teste['roc_auc']:.4f} ({rm.tempo_s:.0f}s)")
    return res

## 5. Diagnóstico de viés e variância e a escolha Random Forest x Gradient Boosting

Diagnóstico de viés (underfitting) e variância (overfitting) e a escolha RF x GB.

Ideia central: Random Forest e Gradient Boosting combinam as mesmas árvores de decisão,
mas atacam problemas opostos.

* Random Forest (bagging): treina muitas árvores profundas em amostras bootstrap e tira a média.
  A média reduz a **variância**; o viés fica parecido com o de uma árvore profunda (baixo).
* Gradient Boosting (boosting): soma árvores rasas em sequência, cada uma corrigindo o erro
  residual das anteriores. Isso reduz o **viés**; a variância é controlada por learning_rate,
  profundidade e regularização.

O diagnóstico compara F1 de treino e de validação (CV) dos dois ensembles para saber qual
problema domina o dataset, usa a árvore de decisão isolada como linha de base e deixa a
validação cruzada decidir; no empate técnico, vence o modelo de menor gap (mais estável).

In [ ]:
"""Diagnóstico de viés (underfitting) e variância (overfitting) e a escolha RF x GB.

Ideia central: Random Forest e Gradient Boosting combinam as mesmas árvores de decisão,
mas atacam problemas opostos.

* Random Forest (bagging): treina muitas árvores profundas em amostras bootstrap e tira a média.
  A média reduz a **variância**; o viés fica parecido com o de uma árvore profunda (baixo).
* Gradient Boosting (boosting): soma árvores rasas em sequência, cada uma corrigindo o erro
  residual das anteriores. Isso reduz o **viés**; a variância é controlada por learning_rate,
  profundidade e regularização.

O diagnóstico compara F1 de treino e de validação (CV) dos dois ensembles para saber qual
problema domina o dataset, usa a árvore de decisão isolada como linha de base e deixa a
validação cruzada decidir; no empate técnico, vence o modelo de menor gap (mais estável).
"""


import numpy as np
from sklearn.base import clone
from sklearn.model_selection import learning_curve


ROTULOS = {
    "variancia": "Alta variância (overfitting)",
    "vies": "Alto viés (underfitting)",
    "ambos": "Viés e variância altos",
    "equilibrado": "Equilibrado",
}


def classificar(f1_treino: float, f1_validacao: float) -> str:
    """Classifica um modelo pelo F1 de treino e de validação da CV.

    * gap (treino - validação) > LIMIAR_GAP_VARIANCIA -> alta variância
    * F1 de treino < LIMIAR_TREINO_VIES -> alto viés (o modelo não ajusta nem o treino)
    """
    gap = f1_treino - f1_validacao
    alta_var = gap > LIMIAR_GAP_VARIANCIA
    alto_vies = f1_treino < LIMIAR_TREINO_VIES
    if alta_var and alto_vies:
        return "ambos"
    if alta_var:
        return "variancia"
    if alto_vies:
        return "vies"
    return "equilibrado"


TAMANHOS_CURVA = np.linspace(0.3, 1.0, 5)  # frações do treino usadas nas curvas de aprendizado


def curvas_aprendizado(res: ResultadoDataset, nomes=("Random Forest", "Gradient Boosting"),
                       tamanhos=TAMANHOS_CURVA) -> dict:
    """Curvas de aprendizado (F1 macro de treino e validação x tamanho do treino)."""
    cv = criar_cv(res.y_train)
    curvas = {}
    for nome in nomes:
        if nome not in res.modelos:
            continue
        n, tr, va = learning_curve(
            clone(res.modelos[nome].estimador), res.X_train, res.y_train, cv=cv,
            train_sizes=tamanhos, scoring="f1_macro", shuffle=True, random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        curvas[nome] = {
            "n": n.tolist(),
            "treino": tr.mean(axis=1).tolist(), "treino_std": tr.std(axis=1).tolist(),
            "validacao": va.mean(axis=1).tolist(), "validacao_std": va.std(axis=1).tolist(),
        }
    res.curvas = curvas
    return curvas


def _tendencia(curva: dict) -> float:
    """Quanto o F1 de validação subiu na última metade da curva (positivo = mais dados ajudam)."""
    v = curva["validacao"]
    return v[-1] - v[len(v) // 2]


ROTULOS["misto"] = "Misto (um modelo superajusta, o outro não)"


def _problema(rf: dict, gb: dict) -> str:
    """Problema predominante do dataset, olhando os dois candidatos (RF e GB)."""
    codigos = {rf["codigo"], gb["codigo"]}
    if codigos <= {"variancia", "ambos"}:
        return "ambos" if "ambos" in codigos and "variancia" not in codigos else "variancia"
    if codigos == {"vies"}:
        return "vies"
    if codigos == {"equilibrado"}:
        return "equilibrado"
    return "misto"


def diagnosticar(res: ResultadoDataset) -> dict:
    """Diagnóstico por modelo e justificativa da escolha entre Random Forest e Gradient Boosting.

    1. O problema do dataset é lido nos dois candidatos: se os dois superajustam (gap alto),
       é alta variância; se os dois não ajustam nem o treino, é alto viés.
    2. A árvore de decisão isolada serve de linha de base: mostra quanto cada ensemble
       reduziu o gap (variância) e quanto melhorou a validação.
    3. Decide a validação cruzada. Se a diferença de F1 entre RF e GB for menor que o
       desvio-padrão entre os folds (empate técnico), vence o de menor gap, isto é, o que
       generaliza de forma mais estável.
    """
    por_modelo = {}
    for nome, rm in res.modelos.items():
        tr, va = rm.cv_treino["f1"], rm.cv_validacao["f1"]
        codigo = classificar(tr, va)
        por_modelo[nome] = {"f1_treino": tr, "f1_validacao": va, "gap": tr - va,
                            "codigo": codigo, "rotulo": ROTULOS[codigo]}

    rf, gb = por_modelo.get("Random Forest"), por_modelo.get("Gradient Boosting")
    base = por_modelo.get("Decision Tree")
    if not (rf and gb):
        res.diagnostico = {"por_modelo": por_modelo, "problema": "equilibrado",
                           "problema_rotulo": ROTULOS["equilibrado"], "escolha_rf_gb": None,
                           "justificativa": ""}
        return res.diagnostico

    problema = _problema(rf, gb)
    std = max(res.modelos["Random Forest"].cv_validacao_std["f1"],
              res.modelos["Gradient Boosting"].cv_validacao_std["f1"])
    diff = rf["f1_validacao"] - gb["f1_validacao"]
    empate = abs(diff) < std
    if empate:
        escolha = "Random Forest" if rf["gap"] <= gb["gap"] else "Gradient Boosting"
    else:
        escolha = "Random Forest" if diff > 0 else "Gradient Boosting"
    outro = "Gradient Boosting" if escolha == "Random Forest" else "Random Forest"
    e, o = por_modelo[escolha], por_modelo[outro]

    def f(x):
        return f"{x:.3f}"

    partes = []
    if problema == "variancia":
        partes.append(
            f"O dataset tem **alta variância**: os dois ensembles ajustam o treino muito melhor que a validação "
            f"(gap de {f(rf['gap'])} no Random Forest e {f(gb['gap'])} no Gradient Boosting). "
            "Na teoria, é o caso do Random Forest, cuja média de árvores reduz variância."
        )
    elif problema == "vies":
        partes.append(
            f"O dataset tem **alto viés**: nem no treino os ensembles passam de F1 {f(max(rf['f1_treino'], gb['f1_treino']))}. "
            "Na teoria, é o caso do Gradient Boosting, que reduz viés corrigindo os erros em sequência."
        )
    elif problema == "ambos":
        partes.append(
            "O dataset tem **viés e variância altos**: os ensembles nem ajustam bem o treino nem generalizam. "
            "Nenhum dos dois resolve isso sozinho; faltam atributos que expliquem o alvo."
        )
    elif problema == "misto":
        sup = rf if rf["codigo"] in ("variancia", "ambos") else gb
        nome_sup = "Random Forest" if sup is rf else "Gradient Boosting"
        partes.append(
            f"O diagnóstico é **misto**: o {nome_sup} superajusta (gap {f(sup['gap'])}), o outro não "
            f"(gap {f(gb['gap'] if sup is rf else rf['gap'])})."
        )
    else:
        partes.append(
            f"O dataset está **equilibrado**: os dois ensembles generalizam bem (gap de {f(rf['gap'])} no "
            f"Random Forest e {f(gb['gap'])} no Gradient Boosting), sem sinal de viés nem de variância altos."
        )

    if base:
        partes.append(
            f"Como linha de base, a árvore de decisão isolada teve F1 de validação {f(base['f1_validacao'])} "
            f"(gap {f(base['gap'])}); o Random Forest chegou a {f(rf['f1_validacao'])} e o Gradient Boosting "
            f"a {f(gb['f1_validacao'])}."
        )

    if empate:
        partes.append(
            f"A diferença de F1 entre os dois ({f(abs(diff))}) é menor que o desvio-padrão entre os folds "
            f"({f(std)}), um empate técnico. O critério de desempate é a estabilidade: o **{escolha}** "
            f"tem o menor gap ({f(e['gap'])} contra {f(o['gap'])})."
        )
    else:
        partes.append(
            f"O **{escolha}** generalizou melhor na validação cruzada (F1 {f(e['f1_validacao'])} contra "
            f"{f(o['f1_validacao'])}, diferença maior que o desvio-padrão entre os folds, {f(std)})."
        )

    teoria = {"variancia": "Random Forest", "vies": "Gradient Boosting"}.get(problema)
    if teoria and teoria == escolha:
        partes.append("A evidência confirma a teoria.")
    elif teoria:
        partes.append(
            f"A evidência contraria a teoria: com os hiperparâmetros encontrados (taxa de aprendizado, "
            f"profundidade, regularização), o {escolha} controlou o problema melhor que o {teoria} neste dataset."
        )

    if max(rf["f1_validacao"], gb["f1_validacao"]) < 0.5:
        partes.append(
            "O F1 de validação baixo, mesmo com o treino quase perfeito, mostra que estes atributos explicam "
            "pouco do alvo: trocar de modelo não muda muito o resultado."
        )

    if res.curvas.get(escolha):
        t = _tendencia(res.curvas[escolha])
        if t > 0.01:
            partes.append(
                f"A curva de aprendizado do {escolha} ainda sobe no fim (+{f(t)} de F1 na segunda metade): "
                "mais exemplos tendem a reduzir a variância restante."
            )
        else:
            partes.append(
                f"A curva de aprendizado do {escolha} já está estável: mais dados do mesmo tipo ajudariam pouco."
            )

    res.diagnostico = {"por_modelo": por_modelo, "problema": problema, "problema_rotulo": ROTULOS[problema],
                       "escolha_rf_gb": escolha, "empate": empate, "justificativa": " ".join(partes)}
    return res.diagnostico

## 6. Gráficos

Gráficos: matrizes de confusão, curvas ROC, curvas de aprendizado e comparativos.

Cada modelo tem sempre a mesma cor em todos os gráficos (paleta categórica validada para
daltonismo); magnitudes usam uma única rampa azul, do claro ao escuro.

In [ ]:
"""Gráficos: matrizes de confusão, curvas ROC, curvas de aprendizado e comparativos.

Cada modelo tem sempre a mesma cor em todos os gráficos (paleta categórica validada para
daltonismo); magnitudes usam uma única rampa azul, do claro ao escuro.
"""


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap


CORES = {
    "Decision Tree": "#2a78d6",
    "Random Forest": "#eb6834",
    "Gradient Boosting": "#1baf7a",
    "SVM": "#eda100",
}
TEXTO, TEXTO_2, GRADE, SUPERFICIE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
AZUL = LinearSegmentedColormap.from_list(
    "azul", ["#f4f8fe", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])


def aplicar_estilo() -> None:
    plt.rcParams.update({
        "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE, "savefig.facecolor": SUPERFICIE,
        "axes.edgecolor": GRADE, "axes.labelcolor": TEXTO_2, "axes.titlecolor": TEXTO,
        "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.8,
        "xtick.color": TEXTO_2, "ytick.color": TEXTO_2, "xtick.labelsize": 9, "ytick.labelsize": 9,
        "legend.frameon": False, "legend.fontsize": 9, "font.size": 10,
        "lines.linewidth": 2, "figure.dpi": 110,
    })


aplicar_estilo()


def grafico_matrizes_confusao(res: ResultadoDataset):
    """Uma matriz por modelo, normalizada por linha (proporção de cada classe real)."""
    n = len(res.modelos)
    muitas = len(res.classes) > 12
    lado = 6.2 if muitas else 4.2
    fig, axes = plt.subplots(1, n, figsize=(lado * n, lado + 0.6), squeeze=False)
    for ax, (nome, rm) in zip(axes[0], res.modelos.items()):
        cm = rm.matriz_confusao
        norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
        ax.imshow(norm, cmap=AZUL, vmin=0, vmax=1)
        rotulos = [str(c) for c in rm.estimador.classes_]
        ax.set_xticks(range(len(rotulos)), rotulos, rotation=90 if muitas or len(rotulos) > 4 else 0,
                      fontsize=6 if muitas else 9)
        ax.set_yticks(range(len(rotulos)), rotulos, fontsize=6 if muitas else 9)
        ax.grid(False)
        if not muitas:
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=9,
                            color="white" if norm[i, j] > 0.55 else TEXTO)
        ax.set_title(f"{nome}\nF1 {rm.teste['f1']:.3f}")
        ax.set_xlabel("Previsto")
        ax.set_ylabel("Real")
    fig.suptitle(f"Matrizes de confusão (teste): {res.nome}", fontsize=14, color=TEXTO, y=1.0)
    fig.tight_layout()
    return fig


def grafico_roc(res: ResultadoDataset):
    multiclasse = len(res.classes) > 2
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    ax.plot([0, 1], [0, 1], ls="--", lw=1, color="#9a9893", label="Aleatório (AUC 0,5)")
    for nome, rm in res.modelos.items():
        fpr, tpr, a = curva_roc(res.y_test, rm.scores_teste, rm.estimador.classes_)
        ax.plot(fpr, tpr, color=CORES.get(nome), label=f"{nome} (AUC {a:.3f})")
    ax.set_xlabel("Taxa de falsos positivos")
    ax.set_ylabel("Taxa de verdadeiros positivos")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    sub = "micro-média um-contra-todos" if multiclasse else "classe positiva"
    ax.set_title(f"Curva ROC (teste, {sub}): {res.nome}")
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig


def grafico_aprendizado(res: ResultadoDataset):
    nomes = list(res.curvas)
    fig, axes = plt.subplots(1, len(nomes), figsize=(6 * len(nomes), 4.4), squeeze=False, sharey=True)
    for ax, nome in zip(axes[0], nomes):
        c = res.curvas[nome]
        n = np.array(c["n"])
        cor = CORES.get(nome)
        for chave, estilo, rot in [("treino", "--", "Treino"), ("validacao", "-", "Validação (CV)")]:
            m, s = np.array(c[chave]), np.array(c[f"{chave}_std"])
            ax.plot(n, m, ls=estilo, color=cor, marker="o", ms=5, label=rot)
            ax.fill_between(n, m - s, m + s, color=cor, alpha=0.12, lw=0)
        d = res.diagnostico.get("por_modelo", {}).get(nome, {})
        ax.set_title(f"{nome}: {d.get('rotulo', '')}")
        ax.set_xlabel("Exemplos de treino")
        ax.set_ylabel("F1 macro")
        ax.legend(loc="lower right")
    fig.suptitle(f"Curvas de aprendizado: {res.nome}", fontsize=14, color=TEXTO)
    fig.tight_layout()
    return fig


def grafico_metricas(res: ResultadoDataset):
    """Métricas de teste lado a lado, um grupo por métrica, uma barra por modelo."""
    metricas = list(NOMES_METRICAS)
    nomes = list(res.modelos)
    larg = 0.8 / len(nomes)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    x = np.arange(len(metricas))
    for k, nome in enumerate(nomes):
        v = [res.modelos[nome].teste[m] for m in metricas]
        ax.bar(x + (k - (len(nomes) - 1) / 2) * larg, v, width=larg * 0.9, color=CORES.get(nome),
               label=nome, edgecolor=SUPERFICIE, linewidth=1)
    ax.set_xticks(x, [NOMES_METRICAS[m] for m in metricas])
    ax.set_ylim(0, 1.05)
    ax.grid(axis="x", visible=False)
    ax.set_ylabel("Valor no teste")
    ax.set_title(f"Métricas no conjunto de teste: {res.nome}")
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    fig.tight_layout()
    return fig


def grafico_mapa(tabela: pd.DataFrame, titulo: str):
    """Mapa de calor dataset x modelo, com o valor escrito em cada célula."""
    fig, ax = plt.subplots(figsize=(8.5, 0.55 * len(tabela) + 1.8))
    vmin = max(0.0, float(np.nanmin(tabela.values)) - 0.05)  # escala do menor valor a 1, para contraste
    ax.imshow(tabela.values, cmap=AZUL, vmin=vmin, vmax=1, aspect="auto")
    ax.set_xticks(range(tabela.shape[1]), tabela.columns)
    ax.set_yticks(range(tabela.shape[0]), tabela.index)
    ax.grid(False)
    for i in range(tabela.shape[0]):
        melhor = np.nanargmax(tabela.values[i])
        for j in range(tabela.shape[1]):
            v = tabela.values[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=10,
                    fontweight="bold" if j == melhor else "normal",
                    color="white" if (v - vmin) / (1 - vmin + 1e-9) > 0.55 else TEXTO)
    ax.xaxis.tick_top()
    ax.set_title(titulo, pad=28)
    fig.tight_layout()
    return fig


def grafico_gaps(diag: pd.DataFrame):
    """Gap treino - validação (F1, CV) por modelo e dataset, com o limiar de alta variância."""
    datasets = diag["Dataset"].unique().tolist()
    nomes = list(CORES)
    larg = 0.8 / len(nomes)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    y = np.arange(len(datasets))
    for k, nome in enumerate(nomes):
        sub = diag[diag["Modelo"] == nome].set_index("Dataset").reindex(datasets)
        ax.barh(y + (k - (len(nomes) - 1) / 2) * larg, sub["Gap"], height=larg * 0.9,
                color=CORES[nome], label=nome, edgecolor=SUPERFICIE, linewidth=1)
    ax.axvline(LIMIAR_GAP_VARIANCIA, color=TEXTO_2, ls="--", lw=1)
    ax.text(LIMIAR_GAP_VARIANCIA, len(datasets) - 0.45, " limiar de alta variância", color=TEXTO_2,
            fontsize=8, va="bottom")
    ax.set_yticks(y, datasets)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("F1 de treino - F1 de validação (CV)")
    ax.set_title("Diagnóstico de variância: quanto o modelo perde do treino para a validação")
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.14))
    fig.tight_layout()
    return fig

## 7. Tabelas comparativas, arquivos de saída e painel HTML

Tabelas comparativas, figuras em disco, tabelas em Markdown e o painel HTML.

In [ ]:
"""Tabelas comparativas, figuras em disco, tabelas em Markdown e o painel HTML."""


import base64
import html
import io
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd



# --------------------------------------------------------------------------------------
# Tabelas
# --------------------------------------------------------------------------------------
def tabela_metricas(resultados: dict[str, ResultadoDataset]) -> pd.DataFrame:
    linhas = []
    for nome_ds, res in resultados.items():
        for nome, rm in res.modelos.items():
            linha = {"Dataset": nome_ds, "Modelo": nome}
            linha.update({NOMES_METRICAS[m]: rm.teste[m] for m in METRICAS})
            linha["F1 CV (validação)"] = rm.cv_validacao["f1"]
            linha["F1 CV (desvio)"] = rm.cv_validacao_std["f1"]
            linha["AUC CV (validação)"] = rm.cv_validacao["roc_auc"]
            linha["Tempo (s)"] = rm.tempo_s
            linhas.append(linha)
    return pd.DataFrame(linhas)


def tabela_diagnostico(resultados: dict[str, ResultadoDataset]) -> pd.DataFrame:
    linhas = []
    for nome_ds, res in resultados.items():
        for nome, d in res.diagnostico.get("por_modelo", {}).items():
            linhas.append({"Dataset": nome_ds, "Modelo": nome, "F1 treino (CV)": d["f1_treino"],
                           "F1 validação (CV)": d["f1_validacao"], "Gap": d["gap"],
                           "Diagnóstico": d["rotulo"]})
    return pd.DataFrame(linhas)


def tabela_escolhas(resultados: dict[str, ResultadoDataset]) -> pd.DataFrame:
    linhas = []
    for nome_ds, res in resultados.items():
        d = res.diagnostico
        if not d:
            continue
        melhor = max(res.modelos, key=lambda m: res.modelos[m].cv_validacao["f1"])
        linhas.append({"Dataset": nome_ds, "Problema (RF e GB)": d["problema_rotulo"],
                       "Escolha RF x GB": d["escolha_rf_gb"], "Melhor dos 4 (F1 CV)": melhor,
                       "Justificativa": d["justificativa"]})
    return pd.DataFrame(linhas)


def tabela_hiperparametros(resultados: dict[str, ResultadoDataset]) -> pd.DataFrame:
    linhas = []
    for nome_ds, res in resultados.items():
        for nome, rm in res.modelos.items():
            p = {k.replace("modelo__", "").replace("prep__num__imputacao__strategy", "imputação"): v
                 for k, v in rm.busca.best_params_.items()}
            txt = ", ".join(f"{k}={round(v, 4) if isinstance(v, float) else v}" for k, v in sorted(p.items()))
            linhas.append({"Dataset": nome_ds, "Modelo": nome, "Hiperparâmetros": txt})
    return pd.DataFrame(linhas)


def pivot(df: pd.DataFrame, valor: str) -> pd.DataFrame:
    ordem_ds = list(dict.fromkeys(df["Dataset"]))
    ordem_m = list(dict.fromkeys(df["Modelo"]))
    return df.pivot(index="Dataset", columns="Modelo", values=valor).reindex(index=ordem_ds, columns=ordem_m)


def markdown(df: pd.DataFrame, destacar_max_linha: bool = False, casas: int = 3) -> str:
    """Tabela Markdown sem dependências; opcionalmente põe em negrito o maior valor de cada linha."""
    cab = [str(df.index.name or "")] + [str(c) for c in df.columns]
    linhas = ["| " + " | ".join(cab) + " |", "|" + "|".join(["---"] + [":---:"] * len(df.columns)) + "|"]
    for idx, row in df.iterrows():
        nums = pd.to_numeric(row, errors="coerce")
        mx = nums.max() if destacar_max_linha else None
        cels = []
        for v, n in zip(row, nums):
            if pd.notna(n) and not isinstance(v, str):
                s = f"{n:.{casas}f}"
                cels.append(f"**{s}**" if mx is not None and n == mx else s)
            else:
                cels.append(str(v))
        linhas.append("| " + " | ".join([str(idx)] + cels) + " |")
    return "\n".join(linhas)


# --------------------------------------------------------------------------------------
# Saída em disco
# --------------------------------------------------------------------------------------
def _png(fig) -> bytes:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def salvar(resultados: dict[str, ResultadoDataset], pasta: str | Path = "resultados") -> dict:
    pasta = Path(pasta)
    (pasta / "figuras").mkdir(parents=True, exist_ok=True)

    met, diag = tabela_metricas(resultados), tabela_diagnostico(resultados)
    esc, hip = tabela_escolhas(resultados), tabela_hiperparametros(resultados)
    met.to_csv(pasta / "metricas.csv", index=False)
    diag.to_csv(pasta / "diagnostico.csv", index=False)
    esc.to_csv(pasta / "escolha_rf_gb.csv", index=False)
    hip.to_csv(pasta / "hiperparametros.csv", index=False)

    figuras: dict[str, dict[str, bytes]] = {"geral": {}}
    figuras["geral"]["f1"] = _png(grafico_mapa(pivot(met, "F1"), "F1 macro no teste"))
    figuras["geral"]["auc"] = _png(grafico_mapa(pivot(met, "ROC AUC"), "ROC AUC no teste"))
    figuras["geral"]["acuracia"] = _png(grafico_mapa(pivot(met, "Acurácia"), "Acurácia no teste"))
    if len(diag):
        figuras["geral"]["gaps"] = _png(grafico_gaps(diag))
    for nome_ds, res in resultados.items():
        f = figuras.setdefault(DATASETS.get(nome_ds, {}).get("slug_curto", nome_ds), {})
        f["metricas"] = _png(grafico_metricas(res))
        f["roc"] = _png(grafico_roc(res))
        f["confusao"] = _png(grafico_matrizes_confusao(res))
        if res.curvas:
            f["aprendizado"] = _png(grafico_aprendizado(res))
    for grupo, figs in figuras.items():
        (pasta / "figuras" / grupo).mkdir(parents=True, exist_ok=True)
        for nome, conteudo in figs.items():
            (pasta / "figuras" / grupo / f"{nome}.png").write_bytes(conteudo)

    # Tabelas em Markdown, prontas para colar no README
    partes = ["# Resultados", "", f"Gerado em {datetime.now():%d/%m/%Y %H:%M}.", ""]
    for rotulo in ["Acurácia", "Precisão", "Recall", "F1", "ROC AUC"]:
        partes += [f"## {rotulo} no teste", "", markdown(pivot(met, rotulo), destacar_max_linha=True), ""]
    partes += ["## F1 na validação cruzada (média dos folds)", "",
               markdown(pivot(met, "F1 CV (validação)"), destacar_max_linha=True), ""]
    if len(diag):
        d2 = diag.copy()
        d2["Resumo"] = d2.apply(lambda r: f"{r['Diagnóstico']} (gap {r['Gap']:.3f})", axis=1)
        partes += ["## Diagnóstico de viés e variância", "",
                   markdown(d2.pivot(index="Dataset", columns="Modelo", values="Resumo")
                            .reindex(index=list(dict.fromkeys(d2["Dataset"])), columns=list(CORES))), ""]
        partes += ["## Random Forest x Gradient Boosting", ""]
        for _, r in esc.iterrows():
            partes += [f"**{r['Dataset']}**: {r['Escolha RF x GB']}. {r['Justificativa']}", ""]
    (pasta / "tabelas.md").write_text("\n".join(partes), encoding="utf-8")

    resumo = {
        "gerado_em": datetime.now().isoformat(timespec="seconds"),
        "datasets": {n: {"linhas": r.n_linhas, "atributos": r.n_atributos, "classes": len(r.classes)}
                     for n, r in resultados.items()},
        "metricas": json.loads(met.to_json(orient="records", force_ascii=False)),
        "diagnostico": json.loads(diag.to_json(orient="records", force_ascii=False)),
        "escolhas": json.loads(esc.to_json(orient="records", force_ascii=False)),
    }
    (pasta / "resumo.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding="utf-8")
    (pasta / "painel.html").write_text(painel_html(resultados, met, diag, esc, hip, figuras), encoding="utf-8")
    return {"metricas": met, "diagnostico": diag, "escolhas": esc, "hiperparametros": hip}


# --------------------------------------------------------------------------------------
# Painel HTML (arquivo único, abre em qualquer navegador, sem servidor)
# --------------------------------------------------------------------------------------
def _img(b: bytes | None, alt: str) -> str:
    if not b:
        return ""
    return f'<img alt="{html.escape(alt)}" src="data:image/png;base64,{base64.b64encode(b).decode()}">'


def _tabela_html(df: pd.DataFrame, destacar: list[str] | None = None, casas: int = 3) -> str:
    destacar = destacar or []
    cab = "".join(f"<th>{html.escape(str(c))}</th>" for c in df.columns)
    maximos = {c: df[c].max() for c in destacar if c in df}
    corpo = []
    for _, row in df.iterrows():
        cels = []
        for c, v in row.items():
            if isinstance(v, float):
                cls = ' class="max"' if c in maximos and v == maximos[c] else ""
                cels.append(f'<td{cls} data-num>{v:.{casas}f}</td>')
            else:
                cels.append(f"<td>{html.escape(str(v))}</td>")
        corpo.append("<tr>" + "".join(cels) + "</tr>")
    return f'<div class="tabela"><table><thead><tr>{cab}</tr></thead><tbody>{"".join(corpo)}</tbody></table></div>'


def _md_negrito(txt: str) -> str:
    partes = html.escape(txt).split("**")
    return "".join(f"<strong>{p}</strong>" if i % 2 else p for i, p in enumerate(partes))


CSS = """
:root{--sup:#fcfcfb;--sup2:#f3f2ef;--borda:#e4e3df;--t1:#0b0b0b;--t2:#52514e;--t3:#7a7975;--azul:#2a78d6;
--dt:#2a78d6;--rf:#eb6834;--gb:#1baf7a;--svm:#eda100}
*{box-sizing:border-box}body{margin:0;font:15px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;
color:var(--t1);background:var(--sup)}
header{padding:28px 40px 12px;border-bottom:1px solid var(--borda)}
header h1{margin:0 0 4px;font-size:24px}header p{margin:0;color:var(--t2)}
nav{display:flex;flex-wrap:wrap;gap:6px;padding:14px 40px;border-bottom:1px solid var(--borda);
position:sticky;top:0;background:var(--sup);z-index:2}
nav button{border:1px solid var(--borda);background:var(--sup);color:var(--t1);padding:7px 14px;
border-radius:999px;font:inherit;font-size:14px;cursor:pointer}
nav button[aria-selected=true]{background:var(--t1);color:var(--sup);border-color:var(--t1)}
main{padding:24px 40px 60px;max-width:1500px}
section{display:none}section.ativa{display:block}
h2{font-size:20px;margin:8px 0 6px}h3{font-size:16px;margin:28px 0 8px}
.sub{color:var(--t2);margin:0 0 18px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin:10px 0 18px}
.kpi{border:1px solid var(--borda);border-radius:10px;padding:14px 16px;background:#fff}
.kpi .r{color:var(--t2);font-size:13px}.kpi .v{font-size:26px;font-weight:600;margin-top:2px}
.kpi .d{color:var(--t3);font-size:12px}
.grade2{display:grid;grid-template-columns:1fr 1fr;gap:18px}
@media(max-width:1000px){.grade2{grid-template-columns:1fr}header,nav,main{padding-left:16px;padding-right:16px}}
img{max-width:100%;height:auto;border:1px solid var(--borda);border-radius:10px;background:#fff}
.tabela{overflow-x:auto;border:1px solid var(--borda);border-radius:10px;background:#fff}
table{border-collapse:collapse;width:100%;font-size:14px}
th,td{padding:8px 12px;text-align:left;border-bottom:1px solid var(--borda);white-space:nowrap}
th{background:var(--sup2);color:var(--t2);font-weight:600}td[data-num]{text-align:right;font-variant-numeric:tabular-nums}
tbody tr:hover{background:#eef4fc}td.max{font-weight:700}
.diag{border-left:4px solid var(--azul);background:#fff;border-radius:0 10px 10px 0;padding:14px 18px;
border-top:1px solid var(--borda);border-right:1px solid var(--borda);border-bottom:1px solid var(--borda)}
.chip{display:inline-block;padding:2px 10px;border-radius:999px;background:var(--sup2);font-size:13px;margin-right:6px}
.leg{display:flex;gap:16px;flex-wrap:wrap;font-size:13px;color:var(--t2);margin:4px 0 14px}
.leg span:before{content:"";display:inline-block;width:10px;height:10px;border-radius:3px;margin-right:6px;
background:var(--c);vertical-align:-1px}
footer{color:var(--t3);font-size:13px;padding:0 40px 30px}
"""

JS = """
const bots=[...document.querySelectorAll('nav button')];
function abrir(id){bots.forEach(b=>b.setAttribute('aria-selected',b.dataset.alvo===id));
document.querySelectorAll('section').forEach(s=>s.classList.toggle('ativa',s.id===id));
if(history.replaceState)history.replaceState(null,'','#'+id);}
bots.forEach(b=>b.addEventListener('click',()=>abrir(b.dataset.alvo)));
const inicial=location.hash.slice(1);
abrir(inicial&&document.getElementById(inicial)?inicial:'geral');
window.addEventListener('hashchange',()=>{const h=location.hash.slice(1);if(document.getElementById(h))abrir(h);});
"""


def painel_html(resultados, met, diag, esc, hip, figuras) -> str:
    legenda = '<div class="leg">' + "".join(
        f'<span style="--c:{c}">{html.escape(n)}</span>' for n, c in CORES.items()) + "</div>"
    abas = [("geral", "Visão geral")] + [
        (DATASETS.get(n, {}).get("slug_curto", n), n) for n in resultados]
    nav = "".join(f'<button data-alvo="{i}" aria-selected="false">{html.escape(t)}</button>' for i, t in abas)

    g = figuras["geral"]
    melhor_por_ds = met.loc[met.groupby("Dataset", sort=False)["F1"].idxmax()]
    resumo = melhor_por_ds[["Dataset", "Modelo", "Acurácia", "Precisão", "F1", "ROC AUC"]].rename(
        columns={"Modelo": "Melhor modelo (F1 teste)"})
    secoes = [f"""
<section id="geral"><h2>Visão geral</h2>
<p class="sub">{len(resultados)} datasets, 4 classificadores, busca de hiperparâmetros com validação cruzada
estratificada e avaliação num conjunto de teste separado (20%).</p>
<h3>Melhor modelo por dataset</h3>{_tabela_html(resumo, ["F1", "ROC AUC"])}
<div class="grade2"><div><h3>F1 macro no teste</h3>{_img(g.get('f1'), 'Mapa de calor do F1 por dataset e modelo')}</div>
<div><h3>ROC AUC no teste</h3>{_img(g.get('auc'), 'Mapa de calor da ROC AUC por dataset e modelo')}</div></div>
<h3>Viés e variância</h3>{_img(g.get('gaps'), 'Gap entre F1 de treino e de validação por modelo e dataset')}
{_tabela_html(diag, [])}
<h3>Random Forest x Gradient Boosting</h3>
{''.join(f'<p class="diag"><strong>{html.escape(r["Dataset"])}</strong>: <span class="chip">{html.escape(str(r["Escolha RF x GB"]))}</span>{_md_negrito(r["Justificativa"])}</p>' for _, r in esc.iterrows())}
<h3>Todas as métricas</h3>{_tabela_html(met.drop(columns=["Tempo (s)"]), [])}
</section>"""]

    for nome_ds, res in resultados.items():
        sid = DATASETS.get(nome_ds, {}).get("slug_curto", nome_ds)
        f = figuras.get(sid, {})
        m = met[met["Dataset"] == nome_ds].drop(columns=["Dataset"])
        melhor = m.loc[m["F1"].idxmax()]
        d = res.diagnostico
        cfg = DATASETS.get(nome_ds, {})
        kpis = f"""<div class="kpis">
<div class="kpi"><div class="r">Exemplos / atributos</div><div class="v">{res.n_linhas} / {res.n_atributos}</div>
<div class="d">{len(res.classes)} classes · alvo <code>{html.escape(str(cfg.get('target', '')))}</code></div></div>
<div class="kpi"><div class="r">Melhor modelo (F1 teste)</div><div class="v">{html.escape(melhor['Modelo'])}</div>
<div class="d">F1 {melhor['F1']:.3f} · acurácia {melhor['Acurácia']:.3f}</div></div>
<div class="kpi"><div class="r">Melhor ROC AUC</div><div class="v">{m['ROC AUC'].max():.3f}</div>
<div class="d">{html.escape(m.loc[m['ROC AUC'].idxmax(), 'Modelo'])}</div></div>
<div class="kpi"><div class="r">Diagnóstico (RF e GB)</div><div class="v" style="font-size:18px">{html.escape(d.get('problema_rotulo', ''))}</div>
<div class="d">escolha RF x GB: {html.escape(str(d.get('escolha_rf_gb')))}</div></div></div>"""
        h = hip[hip["Dataset"] == nome_ds].drop(columns=["Dataset"])
        secoes.append(f"""
<section id="{sid}"><h2>{html.escape(nome_ds)}</h2><p class="sub">{html.escape(cfg.get('descricao', ''))}</p>
{kpis}{legenda}
<h3>Métricas (teste e validação cruzada)</h3>{_tabela_html(m.drop(columns=['Tempo (s)']), ['Acurácia', 'Precisão', 'Recall', 'F1', 'ROC AUC', 'F1 CV (validação)'])}
<div class="grade2"><div><h3>Comparativo</h3>{_img(f.get('metricas'), 'Barras das métricas de teste por modelo')}</div>
<div><h3>Curva ROC</h3>{_img(f.get('roc'), 'Curvas ROC dos quatro modelos')}</div></div>
<h3>Matrizes de confusão</h3>{_img(f.get('confusao'), 'Matrizes de confusão dos quatro modelos')}
<h3>Viés e variância: Random Forest x Gradient Boosting</h3>
<p class="diag">{_md_negrito(d.get('justificativa', ''))}</p>
{_img(f.get('aprendizado'), 'Curvas de aprendizado do Random Forest e do Gradient Boosting')}
<h3>Hiperparâmetros escolhidos</h3>{_tabela_html(h)}
</section>""")

    return f"""<!DOCTYPE html><html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Painel: 4 classificadores em 6 datasets</title>
<style>{CSS}</style></head><body>
<header><h1>Painel de resultados: 4 classificadores em 6 datasets</h1>
<p>SVM, Decision Tree, Random Forest e Gradient Boosting · gerado em {datetime.now():%d/%m/%Y %H:%M}</p></header>
<nav role="tablist">{nav}</nav><main>{''.join(secoes)}</main>
<footer>Gerado por <code>python -m classificadores executar</code>. Os números vêm de resultados/metricas.csv.</footer>
<script>{JS}</script></body></html>"""

## 8. Orquestração do experimento

Orquestra o experimento completo: dados -> 4 modelos -> métricas -> diagnóstico -> relatórios.

In [ ]:
"""Orquestra o experimento completo: dados -> 4 modelos -> métricas -> diagnóstico -> relatórios."""


import time
from pathlib import Path



def executar(nomes: list[str] | None = None, n_iter: int = N_ITER, com_curvas: bool = True,
             usar_hist_gb: bool = USAR_HIST_GB, pasta_dados: Path | None = None,
             log=print) -> dict[str, ResultadoDataset]:
    nomes = nomes or list(DATASETS)
    modelos = criar_modelos(usar_hist_gb)
    resultados: dict[str, ResultadoDataset] = {}
    inicio = time.time()
    for nome in nomes:  # o mesmo laço para os 6 datasets
        cfg = DATASETS[nome]
        log(f"\n{'=' * 60}\n{nome}\n{'=' * 60}")
        try:
            raras = classes_removidas(obter_csv(cfg, pasta_dados), cfg)
            X, y = carregar_dataset(nome, pasta_dados)
        except Exception as e:  # noqa: BLE001
            log(f"   ✗ não foi possível carregar: {e}")
            continue
        if len(raras):
            log(f"   ↳ {len(raras)} classes com < {cfg['min_por_classe']} exemplos removidas: {list(raras.index)}")
        log(f"   {X.shape[0]} linhas x {X.shape[1]} atributos | alvo '{cfg['target']}' com {y.nunique()} classes")
        res = avaliar_dataset(nome, X, y, modelos, n_iter=n_iter, log=log)
        if com_curvas:
            curvas_aprendizado(res)
        d = diagnosticar(res)
        log(f"   → diagnóstico (RF e GB): {d['problema_rotulo']} | escolha RF x GB: {d['escolha_rf_gb']}")
        resultados[nome] = res
    log(f"\nConcluído em {(time.time() - inicio) / 60:.1f} min.")
    return resultados

## 9. Execução
Treina e avalia os 4 modelos nos 6 datasets. Para um teste rápido, use `executar(n_iter=2, com_curvas=False)`.

In [ ]:
resultados = executar()

## 10. Tabelas comparativas

In [ ]:
met, diag = tabela_metricas(resultados), tabela_diagnostico(resultados)
for rotulo in ["Acurácia", "Precisão", "Recall", "F1", "ROC AUC"]:
    display(Markdown(f"### {rotulo} no teste"))
    display(pivot(met, rotulo).style.format("{:.3f}").highlight_max(axis=1, color="#cde2fb"))
display(Markdown("### F1 na validação cruzada"))
display(pivot(met, "F1 CV (validação)").style.format("{:.3f}").highlight_max(axis=1, color="#cde2fb"))

## 11. Resultados por dataset
Métricas, curva ROC, matrizes de confusão, curvas de aprendizado e o diagnóstico de viés e variância.

In [ ]:
for nome_ds, res in resultados.items():
    display(Markdown(f"---\n## {nome_ds}"))
    display(met[met["Dataset"] == nome_ds].drop(columns=["Dataset"]).set_index("Modelo")
            .style.format("{:.3f}").highlight_max(axis=0, color="#cde2fb"))
    for figura in [grafico_metricas, grafico_roc, grafico_matrizes_confusao, grafico_aprendizado]:
        if figura is grafico_aprendizado and not res.curvas:
            continue
        figura(res)
        plt.show()
    display(Markdown(f"**Diagnóstico.** {res.diagnostico['justificativa']}"))

## 12. Viés e variância em todos os datasets

In [ ]:
grafico_gaps(diag)
plt.show()
display(diag.style.format({"F1 treino (CV)": "{:.3f}", "F1 validação (CV)": "{:.3f}", "Gap": "{:.3f}"}))
for _, r in tabela_escolhas(resultados).iterrows():
    display(Markdown(f"**{r['Dataset']}**: {r['Escolha RF x GB']}. {r['Justificativa']}"))

## 13. Salvar resultados e baixar
Gera `resultados/` (CSVs, figuras, `tabelas.md` e `painel.html`) e um `.zip` para baixar.

In [ ]:
import shutil
salvar(resultados, "resultados")
shutil.make_archive("resultados", "zip", "resultados")
try:
    from google.colab import files
    files.download("resultados.zip")
except ImportError:
    print("Arquivos salvos em resultados/ e resultados.zip")